# Second portfolio: replication on Prosper Marketplace loans (extraction + analysis in one file)
A second, independent peer-to-peer lender, **real data only**. Run All (a few minutes). Everything is isolated per section; the last cell lists failures and zips results to `Desktop/Q2/outputs_prosper.zip` - send me that zip.

**Data.** `prosperLoanData.csv` (113,937 Prosper loans, originated Nov-2005 to Mar-2014, snapshot March 2014). The notebook looks for `Desktop/Q2/extra_data/prosper/prosperLoanData.csv` and otherwise downloads a public mirror from GitHub and checks its SHA-256 (`fce13408...`). It is the public Prosper snapshot distributed with the Udacity data-analyst course; **state this provenance in the paper and check the terms of use**.

**Design.** 36-month loans with a final outcome (Completed = 0; Charged off / Defaulted = 1), issued 2006 to March 2011 (36-month loans issued later were still running at the snapshot, so resolved ones would be default-enriched). Two temporal designs:

| design | train | calibrate | test |
|---|---|---|---|
| **B (primary)** | 2006-08 | 2009 | 2010, 2011-Q1 |
| **A (stress)** | 2006-07 | 2008 | 2009, 2010, 2011-Q1 |

Prosper relaunched in 2009 under new credit rules and default rates fell sharply, so design A is a deliberate regime break. **Exact label dates** (`ClosedDate`) are available, so label arrival needs no approximation. Profit per loan (investor view) = interest and fees + service fees + collection fees + non-principal recoveries - net principal loss.

In [ ]:
import os, sys, subprocess, importlib, warnings, json, hashlib, time, traceback, contextlib, shutil, urllib.request
from pathlib import Path
def ensure(pkg, mod=None):
    try: importlib.import_module(mod or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
ensure('lightgbm'); ensure('statsmodels'); ensure('scikit-learn', 'sklearn')
import numpy as np, pandas as pd, lightgbm as lgb, matplotlib
matplotlib.use(os.environ.get('MPL_BACKEND', matplotlib.get_backend()))
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.linear_model import LogisticRegression
from statsmodels.stats.proportion import proportion_confint
warnings.filterwarnings('ignore'); plt.style.use('seaborn-v0_8-whitegrid')
def find_q2():
    if os.environ.get('Q2_DIR'): return Path(os.environ['Q2_DIR'])
    home = Path.home()
    for d in [home/'Desktop', home/'OneDrive'/'Desktop', *sorted(home.glob('OneDrive*/Desktop'))]:
        if (d/'Q2').is_dir(): return d/'Q2'
Q2 = find_q2(); assert Q2 is not None, "No 'Q2' folder on the Desktop; set Q2 = Path(r'...') here."
OUT = Q2/'outputs_prosper'; TAB, FIG = OUT/'tables', OUT/'figures'; TAB.mkdir(parents=True, exist_ok=True); FIG.mkdir(exist_ok=True)
SEED, ALPHA = 42, 0.10; FAILED = []; T0 = time.time()
@contextlib.contextmanager
def section(name):
    t = time.time(); print(f"\n{'=' * 10} {name} {'=' * 10}")
    try: yield
    except Exception:
        FAILED.append(name); print(f"!!! SECTION FAILED: {name}"); traceback.print_exc()
    finally: print(f"[{name}: {time.time() - t:.0f}s | total {(time.time() - T0) / 60:.1f} min]")
def save(df, name): df.round(5).to_csv(TAB/f'{name}.csv', index=False)
def savefig(name):
    for ext in ('png', 'pdf'): plt.savefig(FIG/f'{name}.{ext}', dpi=180, bbox_inches='tight')
def conf_q(scores, alpha):
    s = np.sort(np.asarray(scores)); n = len(s); k = int(np.ceil((n + 1) * (1 - alpha)))
    return np.inf if k > n else s[k - 1]
def true_scores(p, y): return np.where(y == 1, 1 - p, p)
def conformal_sets(p, y, ca, alpha=ALPHA, nbin=5):
    sc = true_scores(p, y); q = conf_q(sc[ca], alpha); q0, q1 = conf_q(sc[ca & (y == 0)], alpha), conf_q(sc[ca & (y == 1)], alpha)
    edges = np.quantile(p[ca], np.linspace(0, 1, nbin + 1)[1:-1]); b = np.digitize(p, edges)
    qb = np.array([conf_q(sc[ca & (b == k)], alpha) for k in range(nbin)])[b]
    return {'static': np.column_stack([p <= q, (1 - p) <= q]), 'class-conditional': np.column_stack([p <= q0, (1 - p) <= q1]), 'bin-conditional': np.column_stack([p <= qb, (1 - p) <= qb])}
def cov_stats(S, y):
    hit = S[np.arange(len(y)), y]; ap, rj = S[:, 0] & ~S[:, 1], S[:, 1] & ~S[:, 0]
    return dict(cov=hit.mean(), cov_default=hit[y == 1].mean() if (y == 1).any() else np.nan, size=S.sum(1).mean(), share_review=1 - ap.mean() - rj.mean(), share_reject_only=rj.mean())
def tiers(S): ap = S[:, 0] & ~S[:, 1]; rj = S[:, 1] & ~S[:, 0]; return ap, ~(ap | rj), rj
def wilson(k, n): return proportion_confint(int(k), int(n), method='wilson')
print("Q2:", Q2)

In [ ]:
# ---------- 0. extraction: Prosper file -> analysis frame ----------
with section('0. data'):
    PF = Q2/'extra_data'/'prosper'/'prosperLoanData.csv'; PF.parent.mkdir(parents=True, exist_ok=True)
    if not PF.exists():
        url = "https://raw.githubusercontent.com/chukwume/prosperloandata/master/prosperLoanData.csv"; print("downloading", url)
        urllib.request.urlretrieve(url, PF)
    sha = hashlib.sha256(PF.read_bytes()).hexdigest(); print("SHA-256:", sha, "| matches expected:", sha == 'fce13408ac22e8201363e71e1a16d9795305f4aaf8c0f67efcd80424bc5b9e02')
    d = pd.read_csv(PF, low_memory=False); print("raw rows:", len(d))
    dt = lambda c: pd.to_datetime(d[c], format='ISO8601', errors='coerce')
    d['orig'], d['closed'], d['listed'], d['first_cl'] = dt('LoanOriginationDate'), dt('ClosedDate'), dt('ListingCreationDate'), dt('FirstRecordedCreditLine')
    keep = d['LoanStatus'].isin(['Completed', 'Chargedoff', 'Defaulted']) & (d['Term'] == 36) & (d['orig'] >= '2006-01-01') & (d['orig'] <= '2011-03-31')
    f = d[keep].sort_values('orig', kind='stable').reset_index(drop=True); print(f"resolved 36-month loans, issued 2006 to Mar-2011: {len(f):,}  default rate {f['LoanStatus'].ne('Completed').mean():.2%}")
    f['default'] = f['LoanStatus'].ne('Completed').astype(int); f['year'], f['month'] = f['orig'].dt.year, f['orig'].dt.month
    f['tm'] = (f['year'] - 2006) * 12 + f['month']; f['reveal'] = np.maximum(((f['closed'].dt.year - 2006) * 12 + f['closed'].dt.month).values, f['tm'].values + 1)
    f['funded'] = f['LoanOriginalAmount'].astype(float)
    f['profit'] = f['LP_InterestandFees'] + f['LP_ServiceFees'] + f['LP_CollectionFees'] + f['LP_NonPrincipalRecoverypayments'] - f['LP_NetPrincipalLoss']
    f['roi'] = f['profit'] / f['funded']
    gmap = {'AA': 0, 'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'HR': 6}
    f['grade_ord'] = f['ProsperRating (Alpha)'].map(gmap).fillna(f['CreditGrade'].map(gmap))
    f['fico'] = (f['CreditScoreRangeLower'] + f['CreditScoreRangeUpper']) / 2; f['dti'] = f['DebtToIncomeRatio'].clip(0, 1.5)
    f['log_income'] = np.log1p(f['StatedMonthlyIncome'].clip(0, 100000)); f['log_amount'] = np.log1p(f['funded']); f['rate'] = f['BorrowerRate']
    f['homeowner'] = f['IsBorrowerHomeowner'].astype(float); f['employment'] = f['EmploymentStatus'].fillna('NA'); f['purpose'] = f['ListingCategory (numeric)'].astype(str); f['state'] = f['BorrowerState'].fillna('NA')
    f['bankcard_util'] = f['BankcardUtilization'].clip(0, 2); f['log_revbal'] = np.log1p(f['RevolvingCreditBalance'].clip(lower=0)); f['log_amt_delinq'] = np.log1p(f['AmountDelinquent'].clip(lower=0)); f['log_avail_bc'] = np.log1p(f['AvailableBankcardCredit'].clip(lower=0))
    f['credit_hist_yrs'] = (f['listed'] - f['first_cl']).dt.days / 365.25
    for a, b in (('open_lines', 'OpenCreditLines'), ('curr_delinq', 'CurrentDelinquencies'), ('delinq_7y', 'DelinquenciesLast7Years'), ('pub_rec_10y', 'PublicRecordsLast10Years'), ('inq_6m', 'InquiriesLast6Months'),
                 ('total_inq', 'TotalInquiries'), ('total_trades', 'TotalTrades'), ('never_delinq_pct', 'TradesNeverDelinquent (percentage)'), ('curr_lines', 'CurrentCreditLines')): f[a] = f[b]
    f['prior_prosper_loans'] = f['TotalProsperLoans'].fillna(0)
    CORE = ['fico', 'dti', 'log_income', 'log_amount', 'homeowner', 'inq_6m', 'bankcard_util', 'curr_delinq', 'open_lines', 'log_revbal']
    BUREAU = ['log_amt_delinq', 'delinq_7y', 'pub_rec_10y', 'total_inq', 'log_avail_bc', 'total_trades', 'never_delinq_pct', 'credit_hist_yrs', 'curr_lines', 'prior_prosper_loans']
    miss = f[CORE + BUREAU + ['rate', 'grade_ord']].isna().groupby(f['year']).mean().T.round(2); print("\nMissing share by year:"); print(miss.to_string())
    drop = [c for c in CORE + BUREAU if f.loc[f.year <= 2008, c].isna().mean() > 0.5]; CORE = [c for c in CORE if c not in drop]; BUREAU = [c for c in BUREAU if c not in drop]; print("dropped (mostly missing in training years):", drop)
    FS = {'P0 with lender rate': (CORE + ['rate'], ['employment', 'purpose']), 'P1 no lender score': (CORE, ['employment', 'purpose']),
          'P3 applicant-only rich': (CORE + BUREAU, ['employment', 'purpose', 'state']), 'P4 everything': (CORE + BUREAU + ['rate', 'grade_ord'], ['employment', 'purpose', 'state'])}
    Xc = {k: pd.get_dummies(f[n + c], columns=c, drop_first=True) for k, (n, c) in FS.items()}
    y, yr, tm, pi, fund, roi = f['default'].values, f['year'].values, f['tm'].values, f['profit'].values, f['funded'].values, f['roi'].values
    T1 = f.groupby('year').agg(N=('default', 'size'), default_rate=('default', 'mean'), mean_rate=('rate', 'mean'), mean_fico=('fico', 'mean'), mean_dti=('dti', 'mean'), mean_amount=('funded', 'mean'),
                               profit_per_loan=('profit', 'mean'), return_on_funded=('profit', lambda s: s.sum() / f.loc[s.index, 'funded'].sum())).reset_index(); save(T1, 'P1_data_summary_by_year'); print(); print(T1.round(3).to_string(index=False))
    g = f.groupby('default').apply(lambda x: x['profit'].sum() / x['funded'].sum()); print(f"\nprofit per $ funded: repaid {g[0]:+.3f}, default {g[1]:+.3f} -> break-even default probability ~ {g[0] / (g[0] - g[1]):.1%}")
    save(pd.DataFrame([dict(profit_per_dollar_repaid=g[0], profit_per_dollar_default=g[1], breakeven_default_prob=g[0] / (g[0] - g[1]))]), 'P2_breakeven')
    lag = ((f['reveal'] - f['tm']))
    desc = pd.DataFrame({'outcome': np.where(y == 1, 'default', 'repaid'), 'months_to_label': lag}).groupby('outcome')['months_to_label'].describe(percentiles=[.1, .25, .5, .75, .9]).reset_index(); save(desc, 'P3_label_lag_by_outcome'); print("\nMonths from origination to label known (exact ClosedDate):"); print(desc.round(1).to_string(index=False))

In [ ]:
# ---------- 1. model zoo and temporal designs ----------
LGB_REG = dict(n_estimators=250, learning_rate=0.03, num_leaves=15, max_depth=4, min_child_samples=100, subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=10.0, random_state=SEED, verbose=-1)
MODELS = ['LGBM default', 'LGBM regularised', 'Logistic (L2)']
def fit_predict(kind, X, mtr):
    if kind == 'LGBM default': m = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=SEED, verbose=-1)
    elif kind == 'LGBM regularised': m = lgb.LGBMClassifier(**LGB_REG)
    else:
        Xf = X.astype('float32'); miss_ = [c for c in Xf.columns if Xf.loc[mtr, c].isna().any()]; ind = Xf[miss_].isna().astype('float32').add_suffix('_na')
        Xf = pd.concat([Xf.fillna(Xf.loc[mtr].median().fillna(0)), ind], axis=1); mu, sd = Xf.loc[mtr].mean().fillna(0), Xf.loc[mtr].std().replace(0, 1).fillna(1); Z = np.nan_to_num(((Xf - mu) / sd).values)
        return LogisticRegression(C=0.1, max_iter=3000).fit(Z[mtr], y[mtr]).predict_proba(Z)[:, 1]
    return m.fit(X[mtr], y[mtr]).predict_proba(X)[:, 1]
def fit_reg(X, target, mtr): return lgb.LGBMRegressor(**LGB_REG).fit(X[mtr], target[mtr]).predict(X)
DESIGNS = {'B': dict(label='B: train 2006-08, calibrate 2009, test 2010-11Q1', train_end=2008, ca=2009, tests=[2010, 2011]),
           'A': dict(label='A: train 2006-07, calibrate 2008, test 2009-11Q1', train_end=2007, ca=2008, tests=[2009, 2010, 2011])}
tlabel = lambda yy: '2011Q1' if yy == 2011 else str(yy)
P = {}; PIPES = {}; ZOO = []
with section('1. model zoo (both designs)'):
    for dk, cfg in DESIGNS.items():
        for fs in FS:
            for kind in MODELS:
                p_val = fit_predict(kind, Xc[fs], yr <= cfg['train_end'] - 1); v = yr == cfg['train_end']; vll = log_loss(y[v], np.clip(p_val[v], 1e-6, 1 - 1e-6))
                p = fit_predict(kind, Xc[fs], yr <= cfg['train_end']); P[(dk, fs, kind)] = p; S = conformal_sets(p, y, yr == cfg['ca']); row = dict(design=dk, feature_set=fs, model=kind, val_logloss=vll)
                for yy in cfg['tests']:
                    m = yr == yy; cs = cov_stats(S['static'][m], y[m]); cm = cov_stats(S['class-conditional'][m], y[m])
                    row.update({f'AUC_{tlabel(yy)}': roc_auc_score(y[m], p[m]), f'gap_pp_{tlabel(yy)}': 100 * (y[m].mean() - p[m].mean()), f'static_cov_{tlabel(yy)}': cs['cov'], f'classcond_cov_default_{tlabel(yy)}': cm['cov_default']})
                ZOO.append(row)
        z = pd.DataFrame([r for r in ZOO if r['design'] == dk]); best = z.loc[z['val_logloss'].idxmin()]
        PIPES[dk] = {'baseline: P0 + LGBM default': ('P0 with lender rate', 'LGBM default'), f'selected: {best.feature_set} + {best.model}': (best.feature_set, best.model)}
    ZOO = pd.DataFrame(ZOO); save(ZOO, 'P4_model_zoo'); pd.set_option('display.width', 260); pd.set_option('display.max_columns', 40)
    for dk in DESIGNS: print(f"\nDesign {DESIGNS[dk]['label']}"); print(ZOO[ZOO.design == dk].drop(columns='design').round(3).to_string(index=False)); print("pipelines:", list(PIPES[dk]))

In [ ]:
# ---------- 2. validity: coverage by test period, controls, alpha ----------
VAL = []; CTRL = []; ALP = []
with section('2. validity'):
    for dk, cfg in DESIGNS.items():
        ca = yr == cfg['ca']
        for pname, key in PIPES[dk].items():
            p = P[(dk,) + key]; S = conformal_sets(p, y, ca)
            for lab, SS in S.items():
                for yy in [cfg['ca']] + cfg['tests']:
                    m = yr == yy; cs = cov_stats(SS[m], y[m]); lo, hi = wilson(cs['cov'] * m.sum(), m.sum())
                    VAL.append(dict(design=dk, pipeline=pname, method=lab, period=tlabel(yy) + (' (calib.)' if yy == cfg['ca'] else ''), N=int(m.sum()), actual_default=y[m].mean(), mean_pred=p[m].mean(), cov=cs['cov'], ci_lo=lo, ci_hi=hi, cov_default=cs['cov_default'], size=cs['size'], share_review=cs['share_review'], share_reject_only=cs['share_reject_only']))
            if pname.startswith('baseline'):
                for ctrl, yy in [(f'within calibration year {cfg["ca"]}', cfg['ca'])] + [(f'within test period {tlabel(t)}', t) for t in cfg['tests']]:
                    idx = np.where(yr == yy)[0]; covs = []
                    for sd in range(50):
                        perm = np.random.default_rng(sd).permutation(idx); h = len(perm) // 2; q = conf_q(true_scores(p[perm[:h]], y[perm[:h]]), ALPHA)
                        S_ = np.column_stack([p[perm[h:]] <= q, (1 - p[perm[h:]]) <= q]); covs.append(S_[np.arange(len(perm) - h), y[perm[h:]]].mean())
                    CTRL.append(dict(design=dk, control=ctrl, mean_cov=np.mean(covs), sd=np.std(covs), repeats=50))
                for a in (0.05, 0.10, 0.20):
                    Sa = conformal_sets(p, y, ca, alpha=a)
                    for lab in ('static', 'class-conditional'):
                        for yy in cfg['tests']:
                            m = yr == yy; cs = cov_stats(Sa[lab][m], y[m]); ALP.append(dict(design=dk, alpha=a, method=lab, period=tlabel(yy), cov=cs['cov'], cov_default=cs['cov_default'], share_reject_only=cs['share_reject_only']))
    VAL, CTRL, ALP = pd.DataFrame(VAL), pd.DataFrame(CTRL), pd.DataFrame(ALP); save(VAL, 'P5_validity'); save(CTRL, 'P6_random_split_controls'); save(ALP, 'P7_alpha_sensitivity')
    print(VAL[VAL.method.isin(['static', 'class-conditional'])].round(3).to_string(index=False)); print(); print(CTRL.round(4).to_string(index=False)); print(); print(ALP.round(3).to_string(index=False))

In [ ]:
# ---------- 3. label timing: exact arrival dates, uniform delays, ACI, rolling recalibration (design B and A) ----------
def aci_reveal(p, y_, month, reveal, cal_scores, alpha, gamma, clip=(0.01, 0.99)):
    s = np.sort(cal_scores); n = len(s); T = len(y_); a = alpha; err = np.zeros(T); size = np.zeros(T); order = np.argsort(reveal, kind='stable'); ptr = 0
    for t in range(T):
        while ptr < T and reveal[order[ptr]] <= month[t] and order[ptr] < t:
            a = np.clip(a + gamma * (alpha - err[order[ptr]]), *clip); ptr += 1
        k = int(np.ceil((n + 1) * (1 - a))); q = -np.inf if k <= 0 else (np.inf if k > n else s[k - 1])
        in0, in1 = p[t] <= q, (1 - p[t]) <= q; err[t] = 1 - (in1 if y_[t] == 1 else in0); size[t] = int(in0) + int(in1)
    return 1 - err, size
def recal_reveal(p, y_, month, reveal, is_test, pool_ok, alpha, window, cal_scores, min_pool=200, horizon=36):
    sc = true_scores(p, y_); T = len(y_); cov = np.full(T, np.nan); size = np.full(T, np.nan)
    for m in np.unique(month[is_test]):
        idx = np.where(is_test & (month == m))[0]; pool = sc[pool_ok & (reveal <= m) & (month >= m - (horizon if window is None else window))]
        if len(pool) < min_pool: pool = cal_scores
        q = conf_q(pool, alpha); S = np.column_stack([p[idx] <= q, (1 - p[idx]) <= q]); cov[idx] = S[np.arange(len(idx)), y_[idx]]; size[idx] = S.sum(1)
    return cov[is_test], size[is_test]
ONL = []; BIAS = []
with section('3. label timing'):
    for dk, cfg in DESIGNS.items():
        sel = np.where(yr >= cfg['ca'])[0]; ys, ms, rv, yrs_ = y[sel], tm[sel], f['reveal'].values[sel], yr[sel]; ca_s, te_s = yrs_ == cfg['ca'], yrs_ > cfg['ca']
        NPM = pd.Series(ms[te_s]).value_counts().mean(); r1 = lambda x: float(f"{x:.1g}"); G = [r1(0.1 / NPM), r1(0.25 / NPM)]; print(f"\nDesign {dk}: loans per test month ~{NPM:.0f}; delayed step sizes {G}")
        for pname, key in PIPES[dk].items():
            p_s = P[(dk,) + key][sel]; cal_sc = true_scores(p_s, ys)[ca_s]; rows = []
            def add(method, arrival, c_, s_):
                for per in [tlabel(t) for t in cfg['tests']] + ['pooled']:
                    m = np.ones(te_s.sum(), bool) if per == 'pooled' else yrs_[te_s] == int(per[:4]); rows.append(dict(design=dk, pipeline=pname, method=method, arrival=arrival, period=per, cov=c_[m].mean(), cov_default=c_[m][ys[te_s][m] == 1].mean(), size=s_[m].mean()))
            S = conformal_sets(p_s, ys, ca_s)['static'][te_s]; add('static (calibration year)', '-', S[np.arange(S.shape[0]), ys[te_s]].astype(float), S.sum(1).astype(float))
            c_, s_ = aci_reveal(p_s[te_s], ys[te_s], ms[te_s], ms[te_s], cal_sc, ALPHA, 0.015); add('ACI gamma=0.015', 'immediate (per loan)', c_, s_)
            for dly in (6, 12):
                for g in G: c_, s_ = aci_reveal(p_s[te_s], ys[te_s], ms[te_s], ms[te_s] + dly, cal_sc, ALPHA, g); add(f'ACI gamma={g:g}', f'uniform {dly} mo', c_, s_)
                c_, s_ = recal_reveal(p_s, ys, ms, ms + dly, te_s, yrs_ >= cfg['ca'], ALPHA, 12, cal_sc); add('rolling recal (W=12)', f'uniform {dly} mo', c_, s_)
            for g in G: c_, s_ = aci_reveal(p_s[te_s], ys[te_s], ms[te_s], rv[te_s], cal_sc, ALPHA, g); add(f'ACI gamma={g:g}', 'exact arrival', c_, s_)
            c_, s_ = recal_reveal(p_s, ys, ms, rv, te_s, yrs_ >= cfg['ca'], ALPHA, None, cal_sc); add('rolling recal, all known labels', 'exact arrival', c_, s_)
            ONL.append(pd.DataFrame(rows))
        for yy in cfg['tests']:                                                                              # label-selection bias at the start of each test year
            snap = (yy - 2006) * 12 + 1; known = (f['reveal'].values <= snap) & (tm <= snap) & (tm >= snap - 36); allw = (tm <= snap) & (tm >= snap - 36)
            BIAS.append(dict(design=dk, snapshot=f'Jan-{yy}', n_window=int(allw.sum()), share_known=known.sum() / allw.sum(), default_rate_known=y[known].mean(), eventual_default_rate=y[allw].mean()))
    ONL = pd.concat(ONL); BIAS = pd.DataFrame(BIAS).drop_duplicates(['snapshot']); save(ONL, 'P8_online_methods'); save(BIAS, 'P9_label_selection_bias')
    print(ONL[ONL.period == 'pooled'].round(3).to_string(index=False)); print(); print(BIAS.round(3).to_string(index=False))

In [ ]:
# ---------- 4. decisions on realised profit, ranking frontier, reviewer breakeven ----------
DEC = []; FRO = []; BE = []
with section('4. decisions'):
    for dk, cfg in DESIGNS.items():
        ca = yr == cfg['ca']; te = np.isin(yr, cfg['tests']); n = te.sum(); pi_te = pi[te]; base_all = pi_te.sum() / n
        blk = pd.factorize(yr[te] * 100 + f['month'].values[te])[0]; NB = blk.max() + 1
        def gain_ci(contrib, B=1000):
            a = np.bincount(blk, contrib[te], NB); A = np.bincount(blk, pi[te], NB); nn = np.bincount(blk, minlength=NB); ix = np.random.default_rng(SEED).integers(0, NB, (B, NB)); dd = (a[ix].sum(1) - A[ix].sum(1)) / nn[ix].sum(1); return np.percentile(dd, [2.5, 97.5])
        print(f"\nDesign {dk}: test loans {n:,}; approve-all profit/applicant ${base_all:,.1f}; return on funded {pi_te.sum() / fund[te].sum():.2%}; default rate {y[te].mean():.2%}")
        R_hat = {k: fit_reg(Xc[k], roi, yr <= cfg['train_end']) for k in ('P0 with lender rate', 'P3 applicant-only rich')}
        for pname, key in PIPES[dk].items():
            p = P[(dk,) + key]; S = conformal_sets(p, y, ca); sc = true_scores(p, y)
            grid = np.quantile(p[ca], np.linspace(0.05, 1.0, 96)); c_best = grid[int(np.argmax([pi[ca & (p <= g)].sum() for g in grid]))]
            best_a, best_v = None, -np.inf
            for a in np.arange(0.01, 0.61, 0.01):
                q = conf_q(sc[ca], a); Sa = np.column_stack([p <= q, (1 - p) <= q]); ap_, rv_, _ = tiers(Sa[ca]); v = pi[ca][ap_].sum()
                if v > best_v: best_a, best_v = a, v
            qa = conf_q(sc[ca], best_a); S_t = np.column_stack([p <= qa, (1 - p) <= qa]); ap_t, rv_t, _ = tiers(S_t)
            pols = [('Approve all', np.ones(len(y), bool), np.zeros(len(y), bool), ['single']), (f'p_hat cutoff <= {c_best:.3f} (tuned on calibration-year profit)', p <= c_best, np.zeros(len(y), bool), ['single']),
                    (f'static conformal, alpha={best_a:.2f} (tuned)', ap_t, rv_t, ['review->reject']), ('Oracle: approve iff profitable', pi > 0, np.zeros(len(y), bool), ['single'])]
            for lab in S: ap, rv, rj = tiers(S[lab]); pols.append((f'{lab} conformal, alpha=0.10', ap, rv, ['review->reject', 'review->approve', 'expert cost 0%', 'expert cost 1%']))
            for pol, ap, rv, rules in pols:
                for rule in rules:
                    if rule in ('single', 'review->reject'): cb, took = np.where(ap, pi, 0.0), ap
                    elif rule == 'review->approve': cb, took = np.where(ap | rv, pi, 0.0), ap | rv
                    else:
                        c = 0.0 if rule.endswith('0%') else 0.01; cb = np.where(ap, pi, 0.0) + np.where(rv, np.where(y == 0, pi, 0.0) - c * fund, 0.0); took = ap | (rv & (y == 0))
                    lo, hi = gain_ci(cb) if pol != 'Approve all' else (0.0, 0.0)
                    DEC.append(dict(design=dk, pipeline=pname, policy=pol, rule=rule, share_approved=took[te].mean(), share_review=rv[te].mean(), profit_per_applicant=cb[te].sum() / n, gain_vs_approve_all=cb[te].sum() / n - base_all, gain_ci_lo=lo, gain_ci_hi=hi,
                                    roi_on_approved=pi[te][took[te]].sum() / max(fund[te][took[te]].sum(), 1), default_rate_approved=y[te][took[te]].mean() if took[te].any() else np.nan))
                    if rule.startswith('review->') or rule.startswith('expert'): pass
            for lab in S:                                                                                    # reviewer break-even (exact, linear in accuracy)
                ap, rv, rj = tiers(S[lab]); tot = dict(rej=pi[te & rj].sum(), pi0=pi[te & rv & (y == 0)].sum(), pi1=pi[te & rv & (y == 1)].sum(), fr=fund[te & rv].sum())
                for cost in (0.0, 0.01):
                    qstar = (tot['rej'] + tot['pi0'] + cost * tot['fr']) / (tot['pi0'] - tot['pi1']) if (tot['pi0'] - tot['pi1']) != 0 else np.nan; bs = []
                    arrs = {k_: np.bincount(blk, v_, NB) for k_, v_ in (('rej', np.where(rj[te], pi[te], 0.)), ('pi0', np.where(rv[te] & (y[te] == 0), pi[te], 0.)), ('pi1', np.where(rv[te] & (y[te] == 1), pi[te], 0.)), ('fr', np.where(rv[te], fund[te], 0.)))}
                    for b in range(1000):
                        pick = np.random.default_rng(b).integers(0, NB, NB); t_ = {k_: v_[pick].sum() for k_, v_ in arrs.items()}; bs.append((t_['rej'] + t_['pi0'] + cost * t_['fr']) / (t_['pi0'] - t_['pi1']))
                    BE.append(dict(design=dk, pipeline=pname, policy=lab, review_cost=cost, share_review=rv[te].mean(), breakeven_accuracy=qstar, ci_lo=np.nanpercentile(bs, 2.5), ci_hi=np.nanpercentile(bs, 97.5)))
        rankers = {'default prob (baseline)': -P[(dk,) + list(PIPES[dk].values())[0]], 'default prob (selected)': -P[(dk,) + list(PIPES[dk].values())[1]], 'expected return, P0': R_hat['P0 with lender rate'], 'expected return, P3': R_hat['P3 applicant-only rich'],
                   "lender's rate (lower=better)": -f['rate'].values.astype(float), "lender's grade (lower=better)": -np.nan_to_num(f['grade_ord'].values.astype(float), nan=9), 'hindsight oracle': pi}
        idx_te = np.where(te)[0]
        for name, s in rankers.items():
            order = idx_te[np.argsort(-s[idx_te], kind='stable')]; cum = np.cumsum(pi[order])
            for sh in (1.0, 0.95, 0.90, 0.80, 0.70, 0.50):
                k = int(round(sh * n)); take = np.zeros(len(y), bool); take[order[:k]] = True; lo, hi = gain_ci(np.where(take, pi, 0.0)) if sh < 1 else (0., 0.)
                FRO.append(dict(design=dk, ranker=name, share_approved=sh, profit_per_applicant=cum[k - 1] / n, gain_vs_approve_all=cum[k - 1] / n - base_all, gain_ci_lo=lo, gain_ci_hi=hi, default_rate_approved=y[order[:k]].mean()))
    DEC, FRO, BE = pd.DataFrame(DEC), pd.DataFrame(FRO), pd.DataFrame(BE); save(DEC, 'P10_decisions'); save(FRO, 'P11_profit_frontier'); save(BE, 'P12_reviewer_breakeven')
    print(DEC[DEC.rule.isin(['single', 'review->reject', 'expert cost 0%', 'expert cost 1%'])].round(3).to_string(index=False)); print(); print(FRO[FRO.share_approved.isin([0.95, 0.90, 0.80])].round(3).to_string(index=False)); print(); print(BE.round(3).to_string(index=False))

In [ ]:
# ---------- 5. training and calibration only on labels known at deployment (exact dates) ----------
DEP = []
with section('5. deployment-realistic training'):
    for dk, cfg in DESIGNS.items():
        T_DEPLOY = (cfg['ca'] - 2006) * 12 + 12; known = f['reveal'].values <= T_DEPLOY; tr_all = yr <= cfg['train_end']; ca_all = yr == cfg['ca']
        for pname, (fs, kind) in PIPES[dk].items():
            for vname, mtr, mca in (('V0 all labels (as in the main design)', tr_all, ca_all), ('V1 train on labels known at deployment, calibrate on all', tr_all & known, ca_all), ('V2 train and calibrate on labels known at deployment', tr_all & known, ca_all & known)):
                p = fit_predict(kind, Xc[fs], mtr); S = conformal_sets(p, y, mca); row = dict(design=dk, pipeline=pname, variant=vname, n_train=int(mtr.sum()), train_default=y[mtr].mean(), n_calib=int(mca.sum()), calib_default=y[mca].mean())
                for yy in cfg['tests']:
                    m = yr == yy; cs = cov_stats(S['static'][m], y[m]); row.update({f'gap_pp_{tlabel(yy)}': 100 * (y[m].mean() - p[m].mean()), f'static_cov_{tlabel(yy)}': cs['cov'], f'static_cov_default_{tlabel(yy)}': cs['cov_default']})
                DEP.append(row)
    DEP = pd.DataFrame(DEP); save(DEP, 'P13_deployment_realistic_training'); print(DEP.round(3).T.to_string())

In [ ]:
# ---------- 6. analytical-property checks (Section 3.6 of the paper) ----------
# Direct checks of the assumptions and predictions of Propositions 1, 2, 4 and 5 on the loan-level predictions.
# Results: tables P14_ties_and_A1, P15_prop5_exact_cut, P16_prop4_random_halves, P17_kappa.
TIES = []; CUT = []; HALVES = []; KAP = []
with section('6. analytical-property checks'):
    rng6 = np.random.default_rng(SEED)
    def _cov_curve(cs_sorted, ts_sorted, grid):
        # static test coverage at each working level, using the conf_q rule (k-th order statistic)
        n = len(cs_sorted); k = np.ceil((n + 1) * (1 - grid)).astype(int)
        q = np.where(k > n, np.inf, cs_sorted[np.clip(k, 1, n) - 1])
        return np.searchsorted(ts_sorted, q, side='right') / len(ts_sorted)
    def kappa_and_abar(cs, ts, h=0.02, step=0.0025):
        # kappa = d(test miscoverage)/d(working level) by local linear fit over [a0-h, a0+h], at a0 = alpha and at alpha-bar
        cs_s, ts_s = np.sort(cs), np.sort(ts); fine = np.arange(0.01, 0.40, 0.0005)
        ab = fine[int(np.argmin(np.abs(_cov_curve(cs_s, ts_s, fine) - 0.90)))]
        def k_at(a0):
            g = np.arange(a0 - h, a0 + h + 1e-12, step); return np.polyfit(g, 1 - _cov_curve(cs_s, ts_s, g), 1)[0]
        return k_at(ALPHA), ab, k_at(max(ab, h + 0.005))
    def kappa_boot(cs, ts, B=300, seed=SEED):
        r = np.random.default_rng(seed); est = kappa_and_abar(cs, ts); bs = []
        for _ in range(B): bs.append(kappa_and_abar(r.choice(cs, len(cs)), r.choice(ts, len(ts))))
        bs = np.array(bs); return est, bs[:, 0].std(ddof=1), bs[:, 2].std(ddof=1)
    for dk, cfg in DESIGNS.items():
        ca = yr == cfg['ca']; te = np.isin(yr, cfg['tests']); n_te = te.sum()
        for pname, key in PIPES[dk].items():
            p = P[(dk,) + key]; sc = true_scores(p, y); q = conf_q(sc[ca], ALPHA); S = conformal_sets(p, y, ca)
            # (a) ties and assumption A1 (Propositions 1, 4, 5)
            for per, m in [(f'{cfg["ca"]} (calib.)', ca), ('test', te)]:
                TIES.append(dict(design=dk, pipeline=pname, period=per, N=int(m.sum()), distinct_p=int(np.unique(p[m]).size), distinct_share=np.unique(p[m]).size / m.sum(),
                                 max_p=p[m].max(), share_p_above_half=(p[m] > 0.5).mean(), q_hat=q, tie_mass_at_q=(np.isclose(sc[m], q)).mean(),
                                 eps0=((p[m] > q) & (y[m] == 0)).sum() / max((y[m] == 0).sum(), 1)))
            # (b) Proposition 5: static "referrals rejected" vs the default-probability ranking cut at the same share
            for lab in ('static', 'class-conditional'):
                ap, rv, rj = tiers(S[lab]); k_ap = int(ap[te].sum()); order = np.argsort(p[te], kind='stable'); top = np.zeros(n_te, bool); top[order[:k_ap]] = True
                same = (top == ap[te]).all(); base = pi[te].sum() / n_te
                g_conf = np.where(ap[te], pi[te], 0).sum() / n_te - base; g_rank = np.where(top, pi[te], 0).sum() / n_te - base
                cut_hi = p[te][ap[te]].max() if k_ap else np.nan; cut_lo = p[te][~ap[te]].min()
                CUT.append(dict(design=dk, pipeline=pname, sets=lab, approved_share=k_ap / n_te, identical_sets=bool(same), gain_conformal=g_conf, gain_ranking_same_share=g_rank,
                                diff_dollars=g_conf - g_rank, max_p_approved=cut_hi, min_p_not_approved=cut_lo, ties_at_cut=int(np.isclose(p[te], cut_lo).sum())))
            if not pname.startswith('baseline'): continue
            # (c) Proposition 4: random halves, 1000 repeats, with and without a tie-breaking jitter on p
            for per, yy in [(f'{cfg["ca"]} (calib.)', cfg['ca'])] + [(tlabel(t), t) for t in cfg['tests']]:
                idx = np.where(yr == yy)[0]; N = len(idx); h_ = N // 2; m_ = N - h_; k_ = int(np.ceil((h_ + 1) * (1 - ALPHA)))
                pred = np.sqrt(k_ * (h_ + 1 - k_) / ((h_ + 1) ** 2 * (h_ + 2)) + ALPHA * (1 - ALPHA) / m_)
                for jit in (False, True):
                    pj = p[idx] + (rng6.uniform(-1e-9, 1e-9, N) if jit else 0); scj = true_scores(pj, y[idx]); covs = []
                    for r in range(1000):
                        perm = np.random.default_rng(10_000 + r).permutation(N); qq = conf_q(scj[perm[:h_]], ALPHA); covs.append((scj[perm[h_:]] <= qq).mean())
                    sd_ = np.std(covs, ddof=1)
                    HALVES.append(dict(design=dk, period=per, N=N, jitter=jit, distinct_p=int(np.unique(p[idx]).size), mean_cov=np.mean(covs), sd=sd_, predicted_sd=pred, ratio=sd_ / pred, ratio_se=1 / np.sqrt(2 * 999)))
            # (d) kappa (Proposition 2): slope of test miscoverage in the working level, at alpha and at alpha-bar
            for per, m in [('calib. random half (benchmark)', None)] + [(tlabel(t), yr == t) for t in cfg['tests']] + [('pooled test', te)]:
                if m is None:
                    idx = np.where(ca)[0]; perm = np.random.default_rng(7).permutation(idx); cal_m = np.zeros(len(y), bool); cal_m[perm[:len(idx) // 2]] = True; tst = np.zeros(len(y), bool); tst[perm[len(idx) // 2:]] = True
                else: cal_m, tst = ca, m
                (k_a, ab, k_ab), se_a, se_ab = kappa_boot(sc[cal_m], sc[tst])
                KAP.append(dict(design=dk, period=per, n_cal=int(cal_m.sum()), n_test=int(tst.sum()), kappa_at_alpha=k_a, kappa_at_alpha_se=se_a, alpha_bar=ab, kappa_at_alpha_bar=k_ab, kappa_at_alpha_bar_se=se_ab))
    TIES, CUT, HALVES, KAP = map(pd.DataFrame, (TIES, CUT, HALVES, KAP))
    save(TIES, 'P14_ties_and_A1'); save(CUT, 'P15_prop5_exact_cut'); save(HALVES, 'P16_prop4_random_halves'); save(KAP, 'P17_kappa')
    for t_, d_ in (('Ties and A1', TIES), ('Proposition 5: exact cut', CUT), ('Proposition 4: random halves (1000 repeats)', HALVES), ('kappa', KAP)):
        print(f"\n{t_}"); print(d_.round(5).to_string(index=False))


In [ ]:
FINAL = []
def add_(t, df_, n=120): FINAL.extend([f"## {t}", df_.round(4).head(n).to_string(index=False), ""])
with section('summary + zip'):
    add_('Data summary', T1); add_('Label lag', desc); add_('Model zoo', ZOO); add_('Validity', VAL[VAL.method.isin(['static', 'class-conditional'])]); add_('Controls', CTRL); add_('Alpha', ALP)
    add_('Online methods (pooled)', ONL[ONL.period == 'pooled']); add_('Label selection bias', BIAS); add_('Decisions', DEC[DEC.rule.isin(['single', 'review->reject', 'expert cost 0%', 'expert cost 1%'])]); add_('Frontier', FRO[FRO.share_approved.isin([0.95, 0.90, 0.80])]); add_('Reviewer breakeven', BE); add_('Deployment-realistic training', DEP); add_('Ties and A1', TIES); add_('Proposition 5 exact cut', CUT); add_('Proposition 4 random halves', HALVES); add_('kappa', KAP)
(OUT/'results_summary.txt').write_text(f"PROSPER REPLICATION ({time.strftime('%Y-%m-%d %H:%M')}), failed sections: {FAILED or 'none'}\n\n" + "\n".join(FINAL))
print(f"FAILED SECTIONS: {FAILED or 'none'}"); print("Send me this file:", shutil.make_archive(str(Q2/'outputs_prosper'), 'zip', OUT))